# Scout 03 — Expected Threat (xT) grid
A project-built grid xT model (Karun Singh style). This is NOT StatsBomb's OBV —
label it honestly everywhere. Writes `statsbomb_marts.mart_expected_threat`.

In [ ]:
PROJECT_ID="statsbomb-football-iq"
from google.colab import auth; auth.authenticate_user()
from google.cloud import bigquery; import numpy as np, pandas as pd
bq=bigquery.Client(project=PROJECT_ID)

In [ ]:
# Pull completed passes+carries with start/end location from core layer
df = bq.query(f'''
  SELECT start_x,start_y,end_x,end_y, "move" AS kind FROM `{PROJECT_ID}.statsbomb_core.fact_pass`
  WHERE completed AND start_x IS NOT NULL AND end_x IS NOT NULL
''').to_dataframe()
GX, GY = 16, 12          # grid: 16 cols x 12 rows over 120x80 pitch
def cell(x,y): return (min(int(x/120*GX),GX-1), min(int(y/80*GY),GY-1))
print(len(df),"moves")

In [ ]:
# Need shots too, to estimate shoot probability + goal prob per cell
shots = bq.query(f'''SELECT start_x,start_y, CAST(is_goal AS INT64) g, xg
                     FROM `{PROJECT_ID}.statsbomb_core.fact_shot`''').to_dataframe()

shoot = np.zeros((GX,GY)); move = np.zeros((GX,GY)); goal = np.zeros((GX,GY))
scount=np.zeros((GX,GY))
for _,r in shots.iterrows():
    cx,cy=cell(r.start_x,r.start_y); shoot[cx,cy]+=1; goal[cx,cy]+=r.g; scount[cx,cy]+=1
for _,r in df.iterrows():
    cx,cy=cell(r.start_x,r.start_y); move[cx,cy]+=1
tot = shoot+move; tot[tot==0]=1
p_shoot = shoot/tot; p_move = move/tot
p_goal  = np.divide(goal, scount, out=np.zeros_like(goal), where=scount>0)

In [ ]:
# Transition matrix: from each cell, where do moves end up?
T = np.zeros((GX,GY,GX,GY))
for _,r in df.iterrows():
    a=cell(r.start_x,r.start_y); b=cell(r.end_x,r.end_y); T[a[0],a[1],b[0],b[1]]+=1
for i in range(GX):
    for j in range(GY):
        s=T[i,j].sum()
        if s>0: T[i,j]/=s

# Iterate xT to convergence
xT=np.zeros((GX,GY))
for _ in range(60):
    newxT=np.zeros((GX,GY))
    for i in range(GX):
        for j in range(GY):
            move_val=(T[i,j]*xT).sum()
            newxT[i,j]=p_shoot[i,j]*p_goal[i,j] + p_move[i,j]*move_val
    xT=newxT
print("xT range:",xT.min().round(4),"->",xT.max().round(4))

In [ ]:
# Write the grid to BigQuery so marts/agent can join xT_added per action
rows=[{"grid_x":i,"grid_y":j,"xt":float(xT[i,j])} for i in range(GX) for j in range(GY)]
bq.load_table_from_dataframe(pd.DataFrame(rows),
    f"{PROJECT_ID}.statsbomb_marts.mart_xt_grid",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
print("xT grid saved. Join on (grid_x,grid_y) to value any action.")